In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
PATH = 'data/Omnichannel_Shopify_Sample.xlsx'
sheet_names = pd.ExcelFile(PATH).sheet_names
print(sheet_names)

['Orders', 'Order_Line_Items', 'Customers', 'Products', 'Inventory', 'Discounts', 'Returns', 'Google_Analytics', 'Meta_Ads', 'Email', 'Google_Ads', 'Display', 'Attribution']


In [3]:
data = pd.ExcelFile(PATH)
orders = pd.read_excel(data, sheet_name='Orders')
order_line_items = pd.read_excel(data, sheet_name='Order_Line_Items')
customers = pd.read_excel(data, sheet_name='Customers')
products = pd.read_excel(data, sheet_name='Products')
inventory = pd.read_excel(data, sheet_name='Inventory')
discounts = pd.read_excel(data, sheet_name='Discounts')
returns = pd.read_excel(data, sheet_name='Returns')
ga4 = pd.read_excel(data, sheet_name='Google_Analytics')
meta = pd.read_excel(data, sheet_name='Meta_Ads')
email = pd.read_excel(data, sheet_name='Email')
ga = pd.read_excel(data, sheet_name='Google_Ads')
display = pd.read_excel(data, sheet_name='Display')
attrib = pd.read_excel(data, sheet_name='Attribution')


In [4]:
customers

,CustomerID,Gender,AgeGroup,State,Acquisition
0,C10001,Female,18-24,NY,Meta
1,C10002,Female,45-54,TX,Google
2,C10003,Female,25-34,NY,Meta
3,C10004,Female,18-24,NY,Organic
4,C10005,Male,35-44,NJ,Email
...,...,...,...,...,...
495,C10496,Male,35-44,NY,Organic
496,C10497,Male,18-24,NY,Organic
497,C10498,Male,35-44,TX,Organic
498,C10499,Female,18-24,TX,Email


In [5]:
orders

,OrderID,Date,CustomerID,SessionID,TotalSales,Promo
0,100001,2024-01-01,C10390,S498078,192.72,SUMMER20
1,100002,2024-01-01,C10336,S970380,52.33,WELCOME10
2,100003,2024-01-01,C10087,S566549,342.76,WELCOME10
3,100004,2024-01-01,C10353,S598614,154.49,SUMMER20
4,100005,2024-01-01,C10458,S359311,246.30,WELCOME10
...,...,...,...,...,...,...
1035,101036,2025-12-22,C10040,S331430,205.56,WELCOME10
1036,101037,2025-12-22,C10429,S216959,88.20,SUMMER20
1037,101038,2025-12-22,C10258,S140591,68.79,NaN
1038,101039,2025-12-22,C10210,S398766,147.44,NaN


In [6]:
total_sales = orders.groupby('CustomerID')['TotalSales'].sum().to_frame()
total_sales

,TotalSales
CustomerID,
C10001,553.87
C10002,243.14
C10003,169.22
C10004,439.04
C10005,129.11
...,...
C10496,84.19
C10497,133.22
C10498,35.44


In [51]:
total_sales = total_sales.sort_values('TotalSales', ascending=False).reset_index()
total_sales['CumShare'] = ((total_sales['TotalSales'].cumsum()/sum(total_sales['TotalSales']))*100).round(2)
total_sales['CustomerShare'] = ((total_sales['TotalSales']/sum(total_sales['TotalSales']))*100).round(2)
total_sales

,CustomerID,TotalSales,CumShare,CustomerShare
0,C10098,1321.47,0.87,0.87
1,C10065,1210.02,1.67,0.80
2,C10273,1121.58,2.41,0.74
3,C10489,998.21,3.06,0.66
4,C10217,993.46,3.72,0.65
...,...,...,...,...
427,C10498,35.44,99.93,0.02
428,C10259,30.98,99.95,0.02
429,C10303,30.37,99.97,0.02
430,C10385,27.54,99.98,0.02


In [55]:
total_orders = orders.groupby(['CustomerID'])['OrderID'].count()
total_sales['NumOrders'] = total_sales['CustomerID'].map(total_orders)
total_sales['AvgSpendPerOrder'] = total_sales['TotalSales']/total_sales['NumOrders']
total_sales

,CustomerID,TotalSales,CumShare,CustomerShare,NumOrders,AvgSpendPerOrder
0,C10098,1321.47,0.87,0.87,6,220.245000
1,C10065,1210.02,1.67,0.80,5,242.004000
2,C10273,1121.58,2.41,0.74,6,186.930000
3,C10489,998.21,3.06,0.66,6,166.368333
4,C10217,993.46,3.72,0.65,5,198.692000
...,...,...,...,...,...,...
427,C10498,35.44,99.93,0.02,1,35.440000
428,C10259,30.98,99.95,0.02,1,30.980000
429,C10303,30.37,99.97,0.02,1,30.370000
430,C10385,27.54,99.98,0.02,1,27.540000


In [62]:
average_purchase_value = sum(total_sales['TotalSales'])/total_sales['TotalSales'].count()
avg_order_size = (total_sales['AvgSpendPerOrder']).mean()
avg_order_size_top_50 = total_sales[total_sales['CumShare']<51]['AvgSpendPerOrder'].mean()
avg_order_size_top_25 = total_sales[total_sales['CumShare']<26]['AvgSpendPerOrder'].mean()
avg_order_size_top_10 = total_sales[total_sales['CumShare']<11]['AvgSpendPerOrder'].mean()
avg_purchase_top_50 = total_sales[total_sales['CumShare']<51]['TotalSales'].mean()
avg_purchase_top_25 = total_sales[total_sales['CumShare']<26]['TotalSales'].mean()
avg_purchase_top_10 = total_sales[total_sales['CumShare']<11]['TotalSales'].mean()


In [63]:
print("Avg Spend per Customer: $", round(average_purchase_value,2))
print("Avg Spend per Customer - Top 50%: $", round(avg_purchase_top_50,2))
print("Avg Spend per Customer - Top 25%: $", round(avg_purchase_top_25,2))
print("Avg Spend per Customer - Top 10%: $", round(avg_purchase_top_10,2))
print("Avg Spend per Order: $", round(avg_order_size,2))
print("Avg Spend per Order - Top 50%: $", round(avg_order_size_top_50,2))
print("Avg Spend per Order - Top 25%: $", round(avg_order_size_top_25,2))
print("Avg Spend per Order - Top 10%: $", round(avg_order_size_top_10,2))


Avg Spend per Customer: $ 351.56
Avg Spend per Customer - Top 50%: $ 652.84
Avg Spend per Customer - Top 25%: $ 785.62
Avg Spend per Customer - Top 10%: $ 938.42
Avg Spend per Order: $ 145.01
Avg Spend per Order - Top 50%: $ 177.83
Avg Spend per Order - Top 25%: $ 189.84
Avg Spend per Order - Top 10%: $ 183.86
